# Vanilla Policy Gradient with a Value Baseline

This notebook is self-contained: upload just this `.ipynb` file to Google Colab and run the cells in order. It uses NumPy, Matplotlib, and IPython; no repository files or downloads are needed.

The Gaussian actor uses fixed nonlinear features, while a linear value baseline estimates Monte Carlo reward-to-go and reduces policy-gradient variance.

## Setup

Set `UPDATES = 2` for a quick check, or keep 3000 for the full experiment.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display

SEED = 7
UPDATES = 3000
BATCH_SIZE = 256
HORIZON = 120
DT = 0.04
GAMMA = 0.97
LEARNING_RATE = 0.03
SIGMA = 0.45
MAX_TORQUE = 12.0
TARGET = np.array([np.pi, 0.0])
EVAL_EPISODES = 64
EVAL_EVERY = 10

BASELINE_REGULARIZATION = 1e-4

## Double-pendulum environment

The vectorized dynamics simulate a batch of fully actuated double pendulums.

In [ ]:
def wrap(angle):
    """Map angles to [-pi, pi)."""
    return (angle + np.pi) % (2 * np.pi) - np.pi


def reset(batch_size, rng):
    """Start a batch near the hanging position."""
    return np.concatenate((rng.normal(0, 0.08, (batch_size, 2)),
                           rng.normal(0, 0.04, (batch_size, 2))), axis=1)


def step(state, torque):
    """Vectorized double-pendulum dynamics for one control interval."""
    # The first axis stores independent trajectories from the same batch.
    angles = state[:, :2].copy()
    velocity = state[:, 2:].copy()
    substep = DT / 4
    # Several semi-implicit Euler substeps improve numerical stability.
    for _ in range(4):
        difference = angles[:, 0] - angles[:, 1]
        sine, cosine = np.sin(difference), np.cos(difference)
        rhs1 = (torque[:, 0] - torque[:, 1] - sine * velocity[:, 1] ** 2
                - 2 * 9.81 * np.sin(angles[:, 0]) - 0.3 * velocity[:, 0])
        rhs2 = (torque[:, 1] + sine * velocity[:, 0] ** 2
                - 9.81 * np.sin(angles[:, 1]) - 0.3 * velocity[:, 1])
        determinant = 2 - cosine ** 2
        acceleration = np.column_stack(((rhs1 - cosine * rhs2) / determinant,
                                         (2 * rhs2 - cosine * rhs1) / determinant))
        velocity += substep * acceleration
        angles = wrap(angles + substep * velocity)

    next_state = np.column_stack((angles, velocity))
    error = wrap(angles - TARGET)
    cost = (np.sum(error ** 2, axis=1) + 0.03 * np.sum(velocity ** 2, axis=1)
            + 0.001 * np.sum(torque ** 2, axis=1))
    return next_state, -cost / HORIZON

## Policy and trajectories

A Gaussian policy samples latent actions, which are mapped to bounded torques. Discounted returns are computed separately for each episode.

In [ ]:
def features(state):
    """Fixed bounded nonlinear basis functions for the policy."""
    # These features are fixed; only the policy weights are learned.
    error = wrap(state[:, :2] - TARGET)
    velocity = np.tanh(state[:, 2:] / 3.0)
    return np.column_stack((
        np.ones(len(state)),
        np.sin(error), np.cos(error), velocity,
        np.sin(error[:, 0] - error[:, 1]),
        np.cos(error[:, 0] - error[:, 1]),
        np.sin(error[:, 0] + error[:, 1]),
        np.cos(error[:, 0] + error[:, 1]),
    ))


N_FEATURES = features(np.zeros((1, 4))).shape[1]


def rollout(weights, rng, batch_size, stochastic=True):
    """Collect on-policy trajectories with weights frozen for the batch."""
    state = reset(batch_size, rng)
    states = np.empty((HORIZON + 1, batch_size, 4))
    rewards = np.empty((HORIZON, batch_size))
    scores = np.empty((HORIZON, batch_size, N_FEATURES, 2))
    states[0] = state
    for time_step in range(HORIZON):
        phi = features(state)
        mean = phi @ weights
        # Sample in an unconstrained latent space, then bound each torque.
        noise = rng.normal(size=mean.shape) if stochastic else np.zeros_like(mean)
        latent_action = mean + SIGMA * noise
        torque = MAX_TORQUE * np.tanh(latent_action)
        # Score of the Gaussian policy with respect to its mean weights.
        scores[time_step] = phi[:, :, None] * (noise / SIGMA)[:, None, :]
        state, rewards[time_step] = step(state, torque)
        states[time_step + 1] = state
    return states, rewards, scores


def reward_to_go(rewards):
    """Compute discounted Monte Carlo returns separately per episode."""
    returns = np.empty_like(rewards)
    running = np.zeros(rewards.shape[1])
    # Backward recursion computes G_t without storing every future sum.
    for time_step in reversed(range(len(rewards))):
        running = rewards[time_step] + GAMMA * running
        returns[time_step] = running
    return returns


def evaluate(weights, stochastic):
    """Evaluate the policy on fixed initial-state samples."""
    # Fixed seeds make different policy checkpoints directly comparable.
    _, rewards, _ = rollout(weights, np.random.default_rng(2026),
                            EVAL_EPISODES, stochastic=stochastic)
    return rewards.sum(axis=0).mean()

## Value baseline and policy update

For every batch, the baseline is fitted to Monte Carlo reward-to-go targets. The actor then uses the advantage $A_t = G_t - V(s_t)$ in the policy-gradient update.

In [ ]:
def fit_baseline(states, returns):
    """Fit V(s) to Monte Carlo reward-to-go targets with ridge regularization."""
    # Use the same fixed features as the policy, but learn separate value weights.
    design = features(states.reshape(-1, 4))
    targets = returns.reshape(-1)
    # Ridge regularization keeps the least-squares system well-conditioned.
    regularizer = BASELINE_REGULARIZATION * np.eye(N_FEATURES)
    baseline_weights = np.linalg.solve(
        design.T @ design + regularizer, design.T @ targets
    )
    prediction = design @ baseline_weights
    return prediction.reshape(returns.shape), baseline_weights


def baseline_policy_gradient(rewards, scores, advantages):
    """Estimate the policy gradient using baseline-subtracted returns."""
    # Subtracting V(s) reduces gradient variance without changing its expectation.
    discount = GAMMA ** np.arange(len(rewards))
    return np.einsum("tb,tbfa,t->fa", advantages, scores, discount) / rewards.shape[1]


def train(updates):
    """Train the actor and return weights plus diagnostic histories."""
    rng = np.random.default_rng(7)
    weights = np.zeros((N_FEATURES, 2))
    train_mean, train_std, objective_history = [], [], []
    baseline_error_history = []
    eval_updates = [0]
    eval_stochastic = [evaluate(weights, True)]
    eval_mean_action = [evaluate(weights, False)]

    for update in range(1, updates + 1):
        # Collect on-policy trajectories with the current actor parameters.
        states, rewards, scores = rollout(weights, rng, BATCH_SIZE)
        returns = reward_to_go(rewards)
        # Fit the baseline on this batch, then compute the policy advantages.
        baseline, _ = fit_baseline(states[:-1], returns)
        advantages = returns - baseline
        # Update the actor by stochastic gradient ascent.
        gradient = baseline_policy_gradient(rewards, scores, advantages)
        weights += LEARNING_RATE * gradient
        if not np.isfinite(weights).all():
            raise FloatingPointError("Nonfinite policy weights")

        episode_rewards = rewards.sum(axis=0)
        train_mean.append(episode_rewards.mean())
        train_std.append(episode_rewards.std())
        objective_history.append(np.mean(
            np.sum(rewards * (GAMMA ** np.arange(HORIZON))[:, None], axis=0)
        ))
        baseline_error_history.append(np.mean((returns - baseline) ** 2))

        # Evaluate periodically without affecting the training random stream.
        if update % EVAL_EVERY == 0 or update == updates:
            eval_updates.append(update)
            eval_stochastic.append(evaluate(weights, True))
            eval_mean_action.append(evaluate(weights, False))
        if update % 50 == 0 or update == updates:
            print(f"Update {update:4d}/{updates} | "
                  f"training reward {train_mean[-1]: .3f} | "
                  f"mean-action evaluation {eval_mean_action[-1]: .3f} | "
                  f"baseline MSE {baseline_error_history[-1]: .4f}")

    history = (np.asarray(train_mean), np.asarray(train_std),
               np.asarray(objective_history), np.asarray(eval_updates),
               np.asarray(eval_stochastic), np.asarray(eval_mean_action),
               np.asarray(baseline_error_history))
    return weights, history

## Training

In [ ]:
print(f"Features: {N_FEATURES} | Batch size: {BATCH_SIZE} | Horizon: {HORIZON}")
print(f"Baseline regularization: {BASELINE_REGULARIZATION}")
learned_weights, training_history = train(UPDATES)
print(f"Final baseline MSE: {training_history[6][-1]:.4f}")

## Reward and baseline diagnostics

In [ ]:
def plot_history(history):
    import matplotlib.pyplot as plt

    train_mean, train_std, objective, eval_updates, eval_stochastic, eval_mean = history
    updates = np.arange(1, len(train_mean) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(updates, train_mean, alpha=0.65, label="Training batch mean")
    axes[0].fill_between(updates, train_mean - train_std,
                         train_mean + train_std, alpha=0.12,
                         label="Training episodes: +/- 1 std")
    axes[0].plot(eval_updates, eval_stochastic, lw=2,
                 label="Stochastic evaluation")
    axes[0].plot(eval_updates, eval_mean, lw=2,
                 label="Mean-action evaluation")
    axes[0].set(xlabel="Policy update", ylabel="Episode reward",
                title="Reward over training")
    axes[0].legend(fontsize=8)
    axes[1].plot(updates, objective)
    axes[1].set(xlabel="Policy update", ylabel="Mean discounted return",
                title="Sampled training objective")
    for axis in axes:
        axis.grid(alpha=0.25)
    fig.tight_layout()
    plt.show()
    plt.close(fig)

In [ ]:
plot_history(training_history[:6])

plt.figure(figsize=(7, 3))
plt.plot(training_history[6])
plt.xlabel("Policy update")
plt.ylabel("Baseline MSE")
plt.title("Value-baseline fit")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## Post-training visualization

The animation compares the initial zero-torque policy with the learned mean-action policy from the same initial state. Use the playback controls below the animation.

In [ ]:
def coordinates(angles):
    """Return Cartesian coordinates for the two links."""
    return (np.array([0, np.sin(angles[0]),
                      np.sin(angles[0]) + np.sin(angles[1])]),
            np.array([0, -np.cos(angles[0]),
                      -np.cos(angles[0]) - np.cos(angles[1])]))


def visualize(weights):
    """Animate the zero-torque and learned mean-action policies."""
    import matplotlib.pyplot as plt
    from matplotlib.animation import FuncAnimation

    # Use the same initial state for both trajectories.
    rng = np.random.default_rng(9001)
    initial_weights = np.zeros_like(weights)
    initial_states, _, _ = rollout(initial_weights, rng, 1, stochastic=False)
    learned_states, _, _ = rollout(weights, np.random.default_rng(9001),
                                   1, stochastic=False)

    fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
    lines = []
    for axis, title in zip(
            axes, ["Initial policy: zero torque", "Learned mean-action policy"]):
        axis.plot(*coordinates(TARGET), "o--", color="gray", alpha=0.6,
                   label="Target")
        line, = axis.plot([], [], "o-", lw=3, markersize=9,
                          label="Pendulum")
        lines.append(line)
        axis.set(xlim=(-2.2, 2.2), ylim=(-2.2, 2.2), title=title,
                 xlabel="x (m)", ylabel="y (m)")
        axis.set_aspect("equal")
        axis.grid(alpha=0.2)
        axis.legend(loc="upper right", fontsize=8)
    clock_text = fig.suptitle("")
    fig.tight_layout()

    def animate(frame):
        for line, trajectory in zip(lines, [initial_states, learned_states]):
            line.set_data(*coordinates(trajectory[frame, 0, :2]))
        clock_text.set_text(f"Time: {frame * DT:.2f} s")
        return [*lines, clock_text]

    # Skip every other frame to keep the animation responsive.
    animation = FuncAnimation(fig, animate, frames=range(0, HORIZON + 1, 2),
                              interval=2000 * DT, blit=False)
    plt.close(fig)
    return animation

In [ ]:
animation = visualize(learned_weights)
display(HTML(animation.to_jshtml()))